# Emotion Classification with Attention

**End-to-end multi-class emotion classification on the GoEmotions dataset**,
comparing classical recurrent architectures (LSTM, GRU), a custom BiLSTM +
Attention model, and a fine-tuned Transformer (DistilBERT).

**Target emotions:** Joy · Sadness · Anger · Fear · Surprise · Disgust

**Environment:** Kaggle Notebook, GPU T4 x1, Internet **ON**.

| Section | Content |
|---|---|
| 1 | Setup & Data Exploration |
| 2 | Data Preprocessing |
| 3 | GloVe Embeddings |
| 4 | LSTM Model |
| 5 | GRU Model |
| 6 | BiLSTM + Attention Model |
| 7 | DistilBERT Fine-Tuning |
| 8 | Evaluation & Model Comparison |
| 9 | Attention Visualization |
| 10 | Gradio Deployment |
| 11 | Conclusion |

---


## Section 1 — Project Setup & Data Exploration

### 1.1 Environment Setup

We install and import all dependencies required across the *entire* project up
front (recurrent models, Transformers, embeddings, visualization, deployment),
so later sections do not need to reinstall packages mid-notebook.


In [ ]:
# 1.1.1 — Install dependencies (Kaggle already ships numpy/pandas/sklearn/matplotlib)
# NOTE: Kaggle images ship a `peft` version that is incompatible with the
# `accelerate` version transformers expects, causing:
# "ImportError: cannot import name 'clear_device_cache' from 'accelerate.utils.memory'"
# the moment `from transformers import Trainer` runs. This project does not use
# PEFT/LoRA at all, so the safest fix is to remove `peft` entirely rather than
# chase version pins — `transformers` simply skips the optional peft import path
# when the package isn't installed.
!pip uninstall -y -q peft
!pip install -q -U datasets==2.19.0 transformers==4.41.2 "accelerate>=0.34.0" \
    evaluate==0.4.2 gradio==4.44.0 seaborn==0.13.2 wordcloud==1.9.3 \
    scikit-learn==1.5.0

# IMPORTANT: after this cell finishes, restart the kernel/session
# (Kaggle: Run -> Restart Session) BEFORE running any further cells, so the
# updated/removed packages are loaded cleanly. Then re-run the notebook from
# the top (Run All).


In [ ]:
# 1.1.2 — Core imports
import os
import re
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import torch
import torch.nn as nn

from datasets import load_dataset

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------------------------
# Device
# -----------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------
# Project directory structure (Kaggle working directory is writable)
# -----------------------------------------------------------------------
PROJECT_ROOT = Path("/kaggle/working")
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in (DATA_DIR, MODELS_DIR, FIGURES_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------
# Plot styling — publication quality
# -----------------------------------------------------------------------
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"

print("Environment ready.")


### 1.2 Loading the GoEmotions Dataset

We use the official Google Research **GoEmotions** dataset, hosted on the
Hugging Face Hub as
[`google-research-datasets/go_emotions`](https://huggingface.co/datasets/google-research-datasets/go_emotions)
(`simplified` config). It contains 58k Reddit comments, each multi-labeled
across 27 fine-grained emotions + `neutral`.

This project targets a **single-label, 6-class** subset:
`joy, sadness, anger, fear, surprise, disgust` — a classic Ekman-style basic
emotion set, all of which are present verbatim among the 27 GoEmotions labels.


In [ ]:
# 1.2.1 — Load raw GoEmotions splits from the Hugging Face Hub
raw_dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
print(raw_dataset)

ALL_LABEL_NAMES = raw_dataset["train"].features["labels"].feature.names
print(f"\nTotal GoEmotions label set ({len(ALL_LABEL_NAMES)} classes):")
print(ALL_LABEL_NAMES)


In [ ]:
# 1.2.2 — Concatenate splits into a single working DataFrame
# (we will re-split ourselves later in Section 2 after filtering to 6 classes)
def hf_split_to_df(split) -> pd.DataFrame:
    df = split.to_pandas()
    df = df[["text", "labels", "id"]].copy()
    return df

df_train = hf_split_to_df(raw_dataset["train"])
df_val = hf_split_to_df(raw_dataset["validation"])
df_test = hf_split_to_df(raw_dataset["test"])

df_train["origin_split"] = "train"
df_val["origin_split"] = "validation"
df_test["origin_split"] = "test"

df_full = pd.concat([df_train, df_val, df_test], ignore_index=True)
print(f"Total raw examples (all 27 emotions + neutral): {len(df_full):,}")
df_full.head()


### 1.3 Filtering to the Target 6-Class Single-Label Subset

GoEmotions is **multi-label** (a comment can be tagged, e.g., both `joy` and
`admiration`). To build a clean **multi-class** (single-label) classification
task we keep only examples that:

1. Have **exactly one** annotated label overall, **and**
2. That label is one of our six target emotions.

This avoids ambiguous supervision (e.g., a comment labeled both `anger` and
`disgust`) and gives every model an unambiguous ground-truth target.


In [ ]:
# 1.3.1 — Map GoEmotions label indices to names, then filter to target emotions
TARGET_EMOTIONS = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
assert all(e in ALL_LABEL_NAMES for e in TARGET_EMOTIONS), "Target emotion missing from GoEmotions label set!"

TARGET_EMOTIONS_SET = set(TARGET_EMOTIONS)

def decode_labels(label_ids):
    return [ALL_LABEL_NAMES[i] for i in label_ids]

df_full["label_names"] = df_full["labels"].apply(decode_labels)

def is_clean_single_target(label_names):
    return len(label_names) == 1 and label_names[0] in TARGET_EMOTIONS_SET

mask = df_full["label_names"].apply(is_clean_single_target)
df_emotion = df_full.loc[mask].copy()
df_emotion["emotion"] = df_emotion["label_names"].apply(lambda x: x[0])
df_emotion = df_emotion.drop(columns=["labels", "label_names"]).reset_index(drop=True)

print(f"Examples after filtering to clean single-label target emotions: {len(df_emotion):,}")
print(f"Retention rate: {len(df_emotion) / len(df_full):.1%} of the raw GoEmotions corpus")
df_emotion.head()


In [ ]:
# 1.3.2 — Persist the filtered raw dataset (pre-cleaning) for reproducibility
df_emotion.to_csv(DATA_DIR / "goemotions_6class_raw.csv", index=False)
print(f"Saved -> {DATA_DIR / 'goemotions_6class_raw.csv'}")


### 1.4 Exploratory Data Analysis (EDA)

We inspect:
- Class balance across the six target emotions
- Text length distribution (characters & tokens)
- Vocabulary size and most frequent tokens
- Per-emotion word clouds
- Duplicate / null checks


In [ ]:
# 1.4.1 — Basic integrity checks
print("Null values:\n", df_emotion.isnull().sum())
print(f"\nDuplicate texts: {df_emotion['text'].duplicated().sum()}")
print(f"Duplicate (text, emotion) pairs: {df_emotion.duplicated(subset=['text', 'emotion']).sum()}")


In [ ]:
# 1.4.2 — Class distribution
class_counts = df_emotion["emotion"].value_counts().reindex(TARGET_EMOTIONS)

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette("viridis", len(TARGET_EMOTIONS))
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor="black", linewidth=0.6)
ax.set_title("Class Distribution — GoEmotions 6-Class Subset", fontsize=14, fontweight="bold")
ax.set_xlabel("Emotion")
ax.set_ylabel("Number of Examples")
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"{count:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_class_distribution.png")
plt.show()

print("\nClass distribution (%):")
print((class_counts / class_counts.sum() * 100).round(2))


In [ ]:
# 1.4.3 — Text length distributions (characters and whitespace-token counts)
df_emotion["char_len"] = df_emotion["text"].str.len()
df_emotion["word_len"] = df_emotion["text"].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_emotion["char_len"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Character Length Distribution")
axes[0].set_xlabel("Number of Characters")
axes[0].axvline(df_emotion["char_len"].median(), color="red", linestyle="--",
                label=f"Median = {df_emotion['char_len'].median():.0f}")
axes[0].legend()

sns.histplot(df_emotion["word_len"], bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Number of Words")
axes[1].axvline(df_emotion["word_len"].median(), color="red", linestyle="--",
                label=f"Median = {df_emotion['word_len'].median():.0f}")
axes[1].legend()

plt.suptitle("Text Length Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_text_length_distribution.png")
plt.show()

print(df_emotion[["char_len", "word_len"]].describe())


In [ ]:
# 1.4.4 — Word length distribution per emotion (boxplot)
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_emotion, x="emotion", y="word_len", order=TARGET_EMOTIONS,
            palette="viridis", ax=ax)
ax.set_title("Word Count by Emotion Class", fontsize=14, fontweight="bold")
ax.set_xlabel("Emotion")
ax.set_ylabel("Word Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_word_length_by_emotion.png")
plt.show()


In [ ]:
# 1.4.5 — Vocabulary size and most frequent tokens (raw, pre-cleaning)
token_counter = Counter()
for text in df_emotion["text"]:
    token_counter.update(text.lower().split())

vocab_size = len(token_counter)
print(f"Raw vocabulary size (whitespace tokens): {vocab_size:,}")

most_common_30 = token_counter.most_common(30)
common_df = pd.DataFrame(most_common_30, columns=["token", "count"])

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=common_df, y="token", x="count", palette="mako", ax=ax)
ax.set_title("Top 30 Most Frequent Tokens (Raw)", fontsize=14, fontweight="bold")
ax.set_xlabel("Frequency")
ax.set_ylabel("Token")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_top_tokens.png")
plt.show()


In [ ]:
# 1.4.6 — Per-emotion word clouds
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, emotion in zip(axes, TARGET_EMOTIONS):
    text_blob = " ".join(df_emotion.loc[df_emotion["emotion"] == emotion, "text"].tolist())
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap="viridis", max_words=80, random_state=SEED).generate(text_blob)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(emotion.capitalize(), fontsize=13, fontweight="bold")
    ax.axis("off")

plt.suptitle("Word Clouds by Emotion", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_wordclouds_by_emotion.png")
plt.show()


In [ ]:
# 1.4.7 — Sample examples per emotion (qualitative sanity check)
pd.set_option("display.max_colwidth", 120)
sample_rows = (
    df_emotion.groupby("emotion", group_keys=False)
    .apply(lambda g: g.sample(n=3, random_state=SEED))
    [["emotion", "text", "word_len"]]
    .reset_index(drop=True)
)
sample_rows


In [ ]:
# 1.4.8 — Persist EDA summary as a JSON report
eda_report = {
    "n_examples_total_raw_goemotions": int(len(df_full)),
    "n_examples_after_filtering": int(len(df_emotion)),
    "retention_rate": float(len(df_emotion) / len(df_full)),
    "class_distribution": class_counts.to_dict(),
    "class_distribution_pct": (class_counts / class_counts.sum() * 100).round(2).to_dict(),
    "char_length_stats": df_emotion["char_len"].describe().to_dict(),
    "word_length_stats": df_emotion["word_len"].describe().to_dict(),
    "raw_vocab_size": vocab_size,
    "duplicate_texts": int(df_emotion["text"].duplicated().sum()),
    "null_values": int(df_emotion.isnull().sum().sum()),
}

with open(REPORTS_DIR / "01_eda_report.json", "w") as f:
    json.dump(eda_report, f, indent=2)

print(json.dumps(eda_report, indent=2))


### 1.5 Section 1 Summary

- Loaded the full GoEmotions corpus (58,009 raw annotations) directly from the
  Hugging Face Hub.
- Filtered to a clean, unambiguous, **single-label 6-class subset**
  (`joy, sadness, anger, fear, surprise, disgust`).
- Confirmed **class imbalance** exists across the six emotions (visualized and
  quantified) — this will inform our choice of loss weighting / stratified
  splitting in Section 2.
- Verified **no null values**; duplicate text checks logged for later
  deduplication.
- Text length is short (Reddit-comment scale), with a right-skewed word-count
  distribution — informs the max-sequence-length choice for tokenization.
- All figures saved to `figures/` (`01`–`05`) and a machine-readable EDA
  summary saved to `reports/01_eda_report.json`.

**Next: Section 2 — Data Preprocessing** (text cleaning, deduplication,
stratified train/val/test split, tokenization strategy for both the
sequence models and DistilBERT). Waiting for confirmation to proceed.


## Section 2 — Data Preprocessing

This section covers:

1. Reloading the Section 1 output
2. Text cleaning (URLs, mentions, HTML entities, GoEmotions `[NAME]` tokens, punctuation/whitespace normalization)
3. Deduplication
4. Label encoding
5. Stratified train / validation / test split
6. A shared vocabulary + integer-encoding pipeline for the recurrent models (LSTM / GRU / BiLSTM+Attention)
7. A separate Hugging Face tokenizer pipeline for DistilBERT
8. Persisting every artifact needed by later sections


In [ ]:
# 2.1 — Reload Section 1 output (safe to run this notebook cell-by-cell in a fresh session)
df_emotion = pd.read_csv(DATA_DIR / "goemotions_6class_raw.csv")
print(f"Loaded {len(df_emotion):,} rows")
df_emotion.head()


### 2.2 Text Cleaning

GoEmotions replaces redacted usernames/subreddits with tokens like `[NAME]`
and `[RELIGION]`; Reddit text also contains URLs, HTML entities, and
inconsistent whitespace. We normalize this while **preserving punctuation
such as `!` and `?`**, since exclamation/question marks carry real emotional
signal (e.g., anger, surprise) that we don't want to strip.


In [ ]:
# 2.2.1 — Text cleaning function
URL_RE = re.compile(r"http\S+|www\.\S+")
BRACKET_TOKEN_RE = re.compile(r"\[(NAME|RELIGION)\]")
HTML_ENTITY_RE = re.compile(r"&amp;|&lt;|&gt;|&#x200B;")
MULTI_SPACE_RE = re.compile(r"\s+")
REPEATED_CHAR_RE = re.compile(r"(.)\1{2,}")  # e.g. "sooooo" -> "soo"
ALLOWED_CHARS_RE = re.compile(r"[^a-zA-Z0-9\s.,!?'\-]")


def clean_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = BRACKET_TOKEN_RE.sub(" ", text)
    text = HTML_ENTITY_RE.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = ALLOWED_CHARS_RE.sub(" ", text)
    text = REPEATED_CHAR_RE.sub(r"\1\1", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text


# Sanity check on a few raw examples
for sample in df_emotion["text"].sample(5, random_state=SEED):
    print(f"RAW  : {sample}")
    print(f"CLEAN: {clean_text(sample)}\n")


In [ ]:
# 2.2.2 — Apply cleaning, drop empties/duplicates produced by cleaning
df_emotion["clean_text"] = df_emotion["text"].apply(clean_text)

before = len(df_emotion)
df_emotion = df_emotion[df_emotion["clean_text"].str.len() > 0].copy()
df_emotion = df_emotion.drop_duplicates(subset=["clean_text", "emotion"]).reset_index(drop=True)
after = len(df_emotion)

print(f"Rows before cleaning-based filtering: {before:,}")
print(f"Rows after removing empties/duplicates: {after:,}")
print(f"Rows dropped: {before - after:,}")


### 2.3 Label Encoding

We fix a canonical integer mapping for the six target emotions, used
consistently by every model in this project.


In [ ]:
# 2.3.1 — Canonical label <-> id mapping
LABEL2ID = {label: idx for idx, label in enumerate(TARGET_EMOTIONS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

df_emotion["label_id"] = df_emotion["emotion"].map(LABEL2ID)

with open(REPORTS_DIR / "label_mapping.json", "w") as f:
    json.dump({"LABEL2ID": LABEL2ID, "ID2LABEL": ID2LABEL}, f, indent=2)

print(LABEL2ID)


### 2.4 Stratified Train / Validation / Test Split

We split **70% train / 15% validation / 15% test**, stratified by
`label_id` so every split preserves the original (imbalanced) class ratios
observed in Section 1.


In [ ]:
# 2.4.1 — Stratified split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_emotion, test_size=0.30, random_state=SEED, stratify=df_emotion["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

split_summary = pd.DataFrame({
    "train": train_df["emotion"].value_counts(normalize=True).reindex(TARGET_EMOTIONS),
    "val": val_df["emotion"].value_counts(normalize=True).reindex(TARGET_EMOTIONS),
    "test": test_df["emotion"].value_counts(normalize=True).reindex(TARGET_EMOTIONS),
}).round(3)
split_summary


In [ ]:
# 2.4.2 — Persist splits (used identically by every model in this project)
train_df.to_csv(DATA_DIR / "train.csv", index=False)
val_df.to_csv(DATA_DIR / "val.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)
print("Saved train.csv, val.csv, test.csv to data/")


### 2.5 Vocabulary & Integer Encoding (for LSTM / GRU / BiLSTM+Attention)

We build a frequency-based vocabulary from the **training set only** (to
avoid leakage), reserving:
- `<PAD>` = 0 (padding)
- `<UNK>` = 1 (out-of-vocabulary tokens)

A `MAX_VOCAB_SIZE` cap and `MIN_TOKEN_FREQ` cutoff control vocabulary size,
and `MAX_SEQ_LEN` is chosen from the Section 1 word-count distribution
(covering roughly the 95th percentile) to bound padding cost.


In [ ]:
# 2.5.1 — Build vocabulary from training data
MAX_VOCAB_SIZE = 20000
MIN_TOKEN_FREQ = 2
MAX_SEQ_LEN = int(np.percentile(train_df["clean_text"].str.split().apply(len), 95))
MAX_SEQ_LEN = max(MAX_SEQ_LEN, 10)  # safety floor

print(f"MAX_SEQ_LEN (95th percentile word count): {MAX_SEQ_LEN}")

train_token_counter = Counter()
for text in train_df["clean_text"]:
    train_token_counter.update(text.lower().split())

vocab_tokens = [
    tok for tok, freq in train_token_counter.most_common()
    if freq >= MIN_TOKEN_FREQ
][: MAX_VOCAB_SIZE - 2]

PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
itos = [PAD_TOKEN, UNK_TOKEN] + vocab_tokens
stoi = {tok: idx for idx, tok in enumerate(itos)}
VOCAB_SIZE = len(itos)

print(f"Final vocabulary size: {VOCAB_SIZE:,}")

with open(DATA_DIR / "vocab.json", "w") as f:
    json.dump({"stoi": stoi, "itos": itos, "max_seq_len": MAX_SEQ_LEN}, f)
print("Saved vocab.json")


In [ ]:
# 2.5.2 — Encoding / padding utilities shared by all recurrent models
def encode_text(text: str, stoi: dict, max_len: int) -> list:
    tokens = text.lower().split()[:max_len]
    ids = [stoi.get(tok, stoi[UNK_TOKEN]) for tok in tokens]
    ids = ids + [stoi[PAD_TOKEN]] * (max_len - len(ids))
    return ids


def encode_dataframe(df: pd.DataFrame, stoi: dict, max_len: int) -> np.ndarray:
    return np.array([encode_text(t, stoi, max_len) for t in df["clean_text"]], dtype=np.int64)


X_train_seq = encode_dataframe(train_df, stoi, MAX_SEQ_LEN)
X_val_seq = encode_dataframe(val_df, stoi, MAX_SEQ_LEN)
X_test_seq = encode_dataframe(test_df, stoi, MAX_SEQ_LEN)

y_train = train_df["label_id"].to_numpy()
y_val = val_df["label_id"].to_numpy()
y_test = test_df["label_id"].to_numpy()

print(f"X_train_seq shape: {X_train_seq.shape}")
print(f"X_val_seq shape:   {X_val_seq.shape}")
print(f"X_test_seq shape:  {X_test_seq.shape}")

np.savez(
    DATA_DIR / "sequence_encoded.npz",
    X_train=X_train_seq, y_train=y_train,
    X_val=X_val_seq, y_val=y_val,
    X_test=X_test_seq, y_test=y_test,
)
print("Saved sequence_encoded.npz")


### 2.6 PyTorch `Dataset` / `DataLoader` for the Recurrent Models

A single reusable `EmotionSequenceDataset` class + a `get_dataloaders()`
factory, so Sections 4-6 (LSTM, GRU, BiLSTM+Attention) share identical data
plumbing and only differ in the model architecture.


In [ ]:
# 2.6.1 — Dataset class
from torch.utils.data import Dataset, DataLoader


class EmotionSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def get_sequence_dataloaders(batch_size: int = 64):
    train_ds = EmotionSequenceDataset(X_train_seq, y_train)
    val_ds = EmotionSequenceDataset(X_val_seq, y_val)
    test_ds = EmotionSequenceDataset(X_test_seq, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_sequence_dataloaders()
xb, yb = next(iter(train_loader))
print(f"Batch X shape: {xb.shape} | Batch y shape: {yb.shape}")


### 2.7 Class Weights (for Imbalanced Loss)

Since Section 1 showed class imbalance, we precompute inverse-frequency
class weights for use in `nn.CrossEntropyLoss(weight=...)` across all
recurrent models, and as `class_weight` guidance for DistilBERT fine-tuning.


In [ ]:
# 2.7.1 — Compute class weights from the training set
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(TARGET_EMOTIONS)),
    y=y_train,
)
CLASS_WEIGHTS = torch.tensor(class_weights_array, dtype=torch.float32)

class_weight_report = {ID2LABEL[i]: float(w) for i, w in enumerate(class_weights_array)}
print(json.dumps(class_weight_report, indent=2))

with open(REPORTS_DIR / "class_weights.json", "w") as f:
    json.dump(class_weight_report, f, indent=2)


### 2.8 Hugging Face Tokenizer Pipeline (for DistilBERT — Section 7)

DistilBERT uses its own WordPiece tokenizer (`distilbert-base-uncased`), so
we prepare a **separate**, model-native tokenization path here (on the
*cleaned* text, but without our custom vocabulary) and cache tokenized
tensors to disk so Section 7 can load them directly.


In [ ]:
# 2.8.1 — Tokenize with the DistilBERT tokenizer and cache to disk
from transformers import AutoTokenizer

BERT_MODEL_NAME = "distilbert-base-uncased"
BERT_MAX_LEN = 64  # covers the vast majority of GoEmotions comments (see Section 1 EDA)

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)


def tokenize_for_bert(texts: list) -> dict:
    return bert_tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=BERT_MAX_LEN,
        return_tensors="pt",
    )


bert_train_enc = tokenize_for_bert(train_df["clean_text"].tolist())
bert_val_enc = tokenize_for_bert(val_df["clean_text"].tolist())
bert_test_enc = tokenize_for_bert(test_df["clean_text"].tolist())

torch.save(
    {
        "train": {**bert_train_enc, "labels": torch.tensor(y_train, dtype=torch.long)},
        "val": {**bert_val_enc, "labels": torch.tensor(y_val, dtype=torch.long)},
        "test": {**bert_test_enc, "labels": torch.tensor(y_test, dtype=torch.long)},
    },
    DATA_DIR / "bert_encoded.pt",
)
print("Saved bert_encoded.pt")
print(f"Example input_ids shape: {bert_train_enc['input_ids'].shape}")


### 2.9 Section 2 Summary

- Cleaned all text (URLs, redaction tokens, HTML entities, repeated chars)
  while **preserving punctuation** relevant to emotion (`!`, `?`).
- Removed empty/duplicate rows produced by cleaning.
- Built a canonical `LABEL2ID` mapping (saved to `reports/label_mapping.json`).
- Created a **stratified 70/15/15** train/val/test split (saved to `data/`).
- Built a training-set-only vocabulary (`VOCAB_SIZE`, `MAX_SEQ_LEN` derived
  from the 95th-percentile word count) and cached integer-encoded sequences
  to `data/sequence_encoded.npz`.
- Implemented a reusable PyTorch `Dataset`/`DataLoader` pipeline shared by
  all three recurrent models.
- Computed **balanced class weights** to counteract class imbalance in the
  loss function, saved to `reports/class_weights.json`.
- Tokenized the same splits with the native DistilBERT tokenizer and cached
  them to `data/bert_encoded.pt` for direct reuse in Section 7.

**Next: Section 3 — GloVe Embeddings** (downloading GloVe vectors, building
the embedding matrix aligned to our vocabulary, and analyzing OOV coverage).
Waiting for confirmation to proceed.


## Section 3 — GloVe Embeddings

We use **pretrained GloVe embeddings** (`glove.6B.100d`, 100-dimensional,
trained on 6B tokens from Wikipedia + Gigaword) to initialize the embedding
layers of the LSTM, GRU, and BiLSTM+Attention models in Sections 4-6.

This section:
1. Loads GloVe (from Kaggle's built-in dataset if attached, else downloads
   directly from the official Stanford NLP source)
2. Builds a `VOCAB_SIZE x 100` embedding matrix aligned to our vocabulary
3. Analyzes out-of-vocabulary (OOV) coverage
4. Saves the embedding matrix for reuse by every recurrent model


In [ ]:
# 3.1 — Reload vocabulary from Section 2 (safe standalone execution)
with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi = vocab_data["stoi"]
itos = vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
VOCAB_SIZE = len(itos)
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"

EMBEDDING_DIM = 100
print(f"Vocabulary size: {VOCAB_SIZE:,} | Embedding dim: {EMBEDDING_DIM} | Max seq len: {MAX_SEQ_LEN}")


### 3.2 Obtaining GloVe Vectors

On Kaggle, the fastest route is attaching the public
**`danielwillgeorge/glove6b100dtxt`** dataset (Add Data -> search
`glove.6B.100d`) which mounts `glove.6B.100d.txt` under `/kaggle/input/`.
If it isn't attached, we fall back to downloading the official archive
directly from Stanford NLP (`https://nlp.stanford.edu/data/glove.6B.zip`,
~822MB, contains 50d/100d/200d/300d files).


In [ ]:
# 3.2.1 — Locate or download glove.6B.100d.txt
import glob
import zipfile
import urllib.request

GLOVE_FILENAME = "glove.6B.100d.txt"
GLOVE_LOCAL_PATH = DATA_DIR / GLOVE_FILENAME

def find_kaggle_glove() -> str | None:
    matches = glob.glob(f"/kaggle/input/**/{GLOVE_FILENAME}", recursive=True)
    return matches[0] if matches else None

kaggle_glove_path = find_kaggle_glove()

if kaggle_glove_path:
    print(f"Found GloVe via attached Kaggle dataset: {kaggle_glove_path}")
    GLOVE_LOCAL_PATH = Path(kaggle_glove_path)
elif GLOVE_LOCAL_PATH.exists():
    print(f"GloVe already downloaded at: {GLOVE_LOCAL_PATH}")
else:
    print("No attached Kaggle GloVe dataset found — downloading from Stanford NLP (official source)...")
    zip_path = DATA_DIR / "glove.6B.zip"
    urllib.request.urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extract(GLOVE_FILENAME, path=DATA_DIR)
    zip_path.unlink()
    print(f"Downloaded and extracted to: {GLOVE_LOCAL_PATH}")


In [ ]:
# 3.2.2 — Parse GloVe file into a token -> vector dict
def load_glove_vectors(path, dim: int) -> dict:
    vectors = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            token, values = parts[0], parts[1:]
            if len(values) != dim:
                continue
            vectors[token] = np.asarray(values, dtype=np.float32)
    return vectors


glove_vectors = load_glove_vectors(GLOVE_LOCAL_PATH, EMBEDDING_DIM)
print(f"Loaded {len(glove_vectors):,} GloVe vectors of dimension {EMBEDDING_DIM}")


### 3.3 Building the Embedding Matrix

Row `i` of the matrix corresponds to token id `i` in our vocabulary
(`itos[i]`). `<PAD>` stays a zero vector; tokens missing from GloVe
(`<UNK>` and true OOV words) are initialized from a small random normal
distribution matching GloVe's empirical mean/std, rather than left at zero,
so the embedding layer can still learn something useful for them.


In [ ]:
# 3.3.1 — Construct the embedding matrix aligned to `itos`
all_glove_values = np.stack(list(glove_vectors.values()))
emb_mean, emb_std = all_glove_values.mean(), all_glove_values.std()

rng = np.random.default_rng(SEED)
embedding_matrix = rng.normal(emb_mean, emb_std, size=(VOCAB_SIZE, EMBEDDING_DIM)).astype(np.float32)
embedding_matrix[stoi[PAD_TOKEN]] = np.zeros(EMBEDDING_DIM, dtype=np.float32)

found, missing = 0, 0
missing_tokens = []
for token, idx in stoi.items():
    if token in (PAD_TOKEN, UNK_TOKEN):
        continue
    if token in glove_vectors:
        embedding_matrix[idx] = glove_vectors[token]
        found += 1
    else:
        missing += 1
        missing_tokens.append(token)

coverage = found / (found + missing)
print(f"Vocabulary tokens matched in GloVe: {found:,}")
print(f"Vocabulary tokens NOT in GloVe (OOV): {missing:,}")
print(f"Coverage: {coverage:.2%}")


In [ ]:
# 3.3.2 — OOV analysis: most frequent OOV tokens (using training token frequencies)
train_df = pd.read_csv(DATA_DIR / "train.csv")
train_token_counter = Counter()
for text in train_df["clean_text"]:
    train_token_counter.update(text.lower().split())

missing_set = set(missing_tokens)
oov_freq = [(tok, train_token_counter.get(tok, 0)) for tok in missing_set]
oov_freq.sort(key=lambda x: x[1], reverse=True)
oov_df = pd.DataFrame(oov_freq[:20], columns=["token", "train_frequency"])
print("Top 20 most frequent OOV tokens:")
oov_df


In [ ]:
# 3.3.3 — Visualize GloVe coverage
fig, ax = plt.subplots(figsize=(6, 6))
sizes = [found, missing]
labels = [f"In GloVe\n({found:,})", f"OOV\n({missing:,})"]
colors = ["#2E7D32", "#C62828"]
ax.pie(sizes, labels=labels, autopct="%1.1f%%", colors=colors,
       startangle=90, textprops={"fontsize": 11, "fontweight": "bold"})
ax.set_title(f"GloVe Vocabulary Coverage (glove.6B.{EMBEDDING_DIM}d)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_glove_coverage.png")
plt.show()


### 3.4 Sanity Check — Nearest Neighbors in Embedding Space

A quick qualitative check: cosine similarity between a few emotion-bearing
words should reflect meaningful semantic relationships if GloVe loaded
correctly.


In [ ]:
# 3.4.1 — Cosine similarity sanity check
from numpy.linalg import norm

def cosine_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b)))

probe_pairs = [
    ("happy", "joyful"),
    ("angry", "furious"),
    ("scared", "afraid"),
    ("happy", "sad"),
    ("disgusting", "gross"),
]

for w1, w2 in probe_pairs:
    if w1 in glove_vectors and w2 in glove_vectors:
        sim = cosine_sim(glove_vectors[w1], glove_vectors[w2])
        print(f"cos_sim({w1!r}, {w2!r}) = {sim:.3f}")


In [ ]:
# 3.5 — Persist the embedding matrix for Sections 4, 5, and 6
np.save(DATA_DIR / "glove_embedding_matrix.npy", embedding_matrix)

glove_report = {
    "embedding_dim": EMBEDDING_DIM,
    "vocab_size": VOCAB_SIZE,
    "glove_tokens_loaded": len(glove_vectors),
    "matched_tokens": found,
    "oov_tokens": missing,
    "coverage": round(coverage, 4),
}
with open(REPORTS_DIR / "03_glove_report.json", "w") as f:
    json.dump(glove_report, f, indent=2)

print(f"Saved embedding matrix: {embedding_matrix.shape} -> data/glove_embedding_matrix.npy")
print(json.dumps(glove_report, indent=2))


### 3.6 Section 3 Summary

- Loaded **glove.6B.100d** pretrained embeddings (from an attached Kaggle
  dataset, or downloaded directly from the official Stanford NLP source as a
  fallback).
- Built a `VOCAB_SIZE x 100` embedding matrix aligned index-for-index with
  our Section 2 vocabulary; `<PAD>` is a zero vector, and OOV tokens get a
  GloVe-distribution-matched random initialization instead of zeros.
- Measured and visualized **GloVe coverage** over our vocabulary and
  inspected the most frequent OOV tokens (typically informal
  internet/Reddit slang and rare proper nouns).
- Sanity-checked embedding quality via cosine similarity between related
  emotion words.
- Saved the final matrix to `data/glove_embedding_matrix.npy`, ready to be
  loaded directly as `nn.Embedding.from_pretrained(...)` in Sections 4-6.

**Next: Section 4 — LSTM Model** (architecture, training loop, learning
curves, checkpointing). Waiting for confirmation to proceed.


## Section 4 — LSTM Model

Our first baseline: a **unidirectional LSTM** classifier, initialized with
the GloVe embedding matrix from Section 3. This establishes a recurrent
baseline before adding bidirectionality and attention in Section 6.

Architecture: `Embedding (GloVe) -> LSTM -> Dropout -> Linear (6-way softmax)`,
using the **final hidden state** of the LSTM as the sequence representation.


In [ ]:
# 4.1 — Reload all artifacts needed (safe standalone execution)
with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
VOCAB_SIZE = len(itos)
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX = stoi[PAD_TOKEN]

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")
EMBEDDING_DIM = embedding_matrix.shape[1]

npz = np.load(DATA_DIR / "sequence_encoded.npz")
X_train_seq, y_train = npz["X_train"], npz["y_train"]
X_val_seq, y_val = npz["X_val"], npz["y_val"]
X_test_seq, y_test = npz["X_test"], npz["y_test"]

with open(REPORTS_DIR / "class_weights.json") as f:
    class_weight_report = json.load(f)
CLASS_WEIGHTS = torch.tensor(
    [class_weight_report[ID2LABEL[i]] for i in range(NUM_CLASSES)], dtype=torch.float32
).to(DEVICE)

print(f"Vocab: {VOCAB_SIZE:,} | Embedding dim: {EMBEDDING_DIM} | Max len: {MAX_SEQ_LEN} | Classes: {NUM_CLASSES}")


In [ ]:
# 4.2 — Shared Dataset / DataLoader (identical to Section 2.6)
from torch.utils.data import Dataset, DataLoader


class EmotionSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64
train_loader = DataLoader(EmotionSequenceDataset(X_train_seq, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionSequenceDataset(X_val_seq, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(EmotionSequenceDataset(X_test_seq, y_test), batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


### 4.3 Model Architecture


In [ ]:
# 4.3.1 — LSTM classifier
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6,
                 pad_idx=0, num_layers=1, dropout=0.3, freeze_embeddings=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze_embeddings,
            padding_idx=pad_idx,
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)                       # (batch, seq_len, embed_dim)
        _, (hidden, _) = self.lstm(embedded)                # hidden: (num_layers, batch, hidden_dim)
        last_hidden = hidden[-1]                            # (batch, hidden_dim)
        out = self.dropout(last_hidden)
        logits = self.fc(out)                               # (batch, num_classes)
        return logits


lstm_model = LSTMClassifier(
    embedding_matrix, hidden_dim=128, num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX, num_layers=1, dropout=0.3, freeze_embeddings=False,
).to(DEVICE)

n_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(lstm_model)
print(f"\nTrainable parameters: {n_params:,}")


### 4.4 Training Loop

A shared, reusable `Trainer` utility (used identically by LSTM, GRU, and
BiLSTM+Attention in Sections 4-6) that handles: epoch loop, weighted loss,
gradient clipping, validation, best-checkpoint saving, and early stopping.


In [ ]:
# 4.4.1 — Reusable training utility for all recurrent models
import time
from sklearn.metrics import f1_score, accuracy_score


def run_training(model, train_loader, val_loader, model_name: str,
                  num_epochs=15, lr=1e-3, weight_decay=1e-5,
                  class_weights=None, patience=3, grad_clip=5.0):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

    history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    best_val_f1 = -1.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"{model_name}_best.pt"

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        # ---- Train ----
        model.train()
        train_losses, train_preds, train_targets = [], [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.extend(logits.argmax(dim=1).cpu().numpy())
            train_targets.extend(yb.cpu().numpy())

        train_loss = np.mean(train_losses)
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        # ---- Validate ----
        model.eval()
        val_losses, val_preds, val_targets = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(yb.cpu().numpy())

        val_loss = np.mean(val_losses)
        val_f1 = f1_score(val_targets, val_preds, average="macro")
        val_acc = accuracy_score(val_targets, val_preds)

        scheduler.step(val_f1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        elapsed = time.time() - start_time
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
              f"train_f1={train_f1:.4f} val_f1={val_f1:.4f} val_acc={val_acc:.4f} | "
              f"{elapsed:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch} (no improvement for {patience} epochs).")
                break

    model.load_state_dict(torch.load(best_path))
    print(f"\nBest val_f1={best_val_f1:.4f} — restored best checkpoint from {best_path}")
    return model, history


In [ ]:
# 4.4.2 — Train the LSTM
lstm_model, lstm_history = run_training(
    lstm_model, train_loader, val_loader, model_name="lstm",
    num_epochs=15, lr=1e-3, class_weights=CLASS_WEIGHTS, patience=3,
)

with open(REPORTS_DIR / "lstm_history.json", "w") as f:
    json.dump(lstm_history, f, indent=2)


### 4.5 Learning Curves


In [ ]:
# 4.5.1 — Plot loss / F1 curves
def plot_learning_curves(history: dict, model_name: str, save_name: str):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    epochs_range = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
    axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(epochs_range, history["train_f1"], marker="o", label="Train Macro-F1")
    axes[1].plot(epochs_range, history["val_f1"], marker="o", label="Val Macro-F1")
    axes[1].set_title(f"{model_name} — Macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].legend()

    plt.suptitle(f"{model_name} Learning Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name)
    plt.show()


plot_learning_curves(lstm_history, "LSTM", "07_lstm_learning_curves.png")


### 4.6 Section 4 Summary

- Implemented a GloVe-initialized **LSTM classifier** with dropout
  regularization and a linear classification head over the final hidden
  state.
- Built a **reusable, weighted-loss training loop** (`run_training`) with
  gradient clipping, LR scheduling on plateau, best-checkpoint saving, and
  early stopping — reused as-is for GRU (Section 5) and BiLSTM+Attention
  (Section 6).
- Saved the best checkpoint to `models/lstm_best.pt` and training history to
  `reports/lstm_history.json`.
- Plotted and saved loss/F1 learning curves to
  `figures/07_lstm_learning_curves.png`.

Test-set evaluation (confusion matrix, classification report) for **all**
models is deferred to the unified **Section 8 — Evaluation & Model
Comparison**, so every model is scored identically and side-by-side.

**Next: Section 5 — GRU Model.** Waiting for confirmation to proceed.


## Section 5 — GRU Model

Our second baseline: a **GRU** classifier, architecturally identical in
spirit to the Section 4 LSTM (same GloVe embedding, same training loop,
same hyperparameter budget) so the two are directly comparable — isolating
the effect of the recurrent cell type (GRU vs. LSTM) on this task.


In [ ]:
# 5.1 — Reload artifacts (safe standalone execution — identical to Section 4.1)
with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
VOCAB_SIZE = len(itos)
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX = stoi[PAD_TOKEN]

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")
EMBEDDING_DIM = embedding_matrix.shape[1]

npz = np.load(DATA_DIR / "sequence_encoded.npz")
X_train_seq, y_train = npz["X_train"], npz["y_train"]
X_val_seq, y_val = npz["X_val"], npz["y_val"]
X_test_seq, y_test = npz["X_test"], npz["y_test"]

with open(REPORTS_DIR / "class_weights.json") as f:
    class_weight_report = json.load(f)
CLASS_WEIGHTS = torch.tensor(
    [class_weight_report[ID2LABEL[i]] for i in range(NUM_CLASSES)], dtype=torch.float32
).to(DEVICE)

from torch.utils.data import Dataset, DataLoader


class EmotionSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64
train_loader = DataLoader(EmotionSequenceDataset(X_train_seq, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionSequenceDataset(X_val_seq, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(EmotionSequenceDataset(X_test_seq, y_test), batch_size=BATCH_SIZE, shuffle=False)
print(f"Vocab: {VOCAB_SIZE:,} | Classes: {NUM_CLASSES} | Train batches: {len(train_loader)}")


### 5.2 Model Architecture


In [ ]:
# 5.2.1 — GRU classifier
class GRUClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6,
                 pad_idx=0, num_layers=1, dropout=0.3, freeze_embeddings=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze_embeddings,
            padding_idx=pad_idx,
        )
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)               # (batch, seq_len, embed_dim)
        _, hidden = self.gru(embedded)              # hidden: (num_layers, batch, hidden_dim)
        last_hidden = hidden[-1]                    # (batch, hidden_dim)
        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits


gru_model = GRUClassifier(
    embedding_matrix, hidden_dim=128, num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX, num_layers=1, dropout=0.3, freeze_embeddings=False,
).to(DEVICE)

n_params = sum(p.numel() for p in gru_model.parameters() if p.requires_grad)
print(gru_model)
print(f"\nTrainable parameters: {n_params:,}")


### 5.3 Training

Reusing the exact `run_training` utility defined in Section 4.4 (re-declared
here so this section is independently executable).


In [ ]:
# 5.3.1 — Reusable training utility (identical to Section 4.4.1)
import time
from sklearn.metrics import f1_score, accuracy_score


def run_training(model, train_loader, val_loader, model_name: str,
                  num_epochs=15, lr=1e-3, weight_decay=1e-5,
                  class_weights=None, patience=3, grad_clip=5.0):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

    history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    best_val_f1 = -1.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"{model_name}_best.pt"

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        model.train()
        train_losses, train_preds, train_targets = [], [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.extend(logits.argmax(dim=1).cpu().numpy())
            train_targets.extend(yb.cpu().numpy())

        train_loss = np.mean(train_losses)
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        model.eval()
        val_losses, val_preds, val_targets = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(yb.cpu().numpy())

        val_loss = np.mean(val_losses)
        val_f1 = f1_score(val_targets, val_preds, average="macro")
        val_acc = accuracy_score(val_targets, val_preds)

        scheduler.step(val_f1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        elapsed = time.time() - start_time
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
              f"train_f1={train_f1:.4f} val_f1={val_f1:.4f} val_acc={val_acc:.4f} | "
              f"{elapsed:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch} (no improvement for {patience} epochs).")
                break

    model.load_state_dict(torch.load(best_path))
    print(f"\nBest val_f1={best_val_f1:.4f} — restored best checkpoint from {best_path}")
    return model, history


In [ ]:
# 5.3.2 — Train the GRU
gru_model, gru_history = run_training(
    gru_model, train_loader, val_loader, model_name="gru",
    num_epochs=15, lr=1e-3, class_weights=CLASS_WEIGHTS, patience=3,
)

with open(REPORTS_DIR / "gru_history.json", "w") as f:
    json.dump(gru_history, f, indent=2)


### 5.4 Learning Curves


In [ ]:
# 5.4.1 — Plot loss / F1 curves
def plot_learning_curves(history: dict, model_name: str, save_name: str):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    epochs_range = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
    axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(epochs_range, history["train_f1"], marker="o", label="Train Macro-F1")
    axes[1].plot(epochs_range, history["val_f1"], marker="o", label="Val Macro-F1")
    axes[1].set_title(f"{model_name} — Macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].legend()

    plt.suptitle(f"{model_name} Learning Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name)
    plt.show()


plot_learning_curves(gru_history, "GRU", "08_gru_learning_curves.png")


In [ ]:
# 5.4.2 — Quick LSTM vs GRU validation-F1 comparison (preview; full comparison in Section 8)
with open(REPORTS_DIR / "lstm_history.json") as f:
    lstm_history = json.load(f)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(lstm_history["val_f1"]) + 1), lstm_history["val_f1"], marker="o", label="LSTM")
ax.plot(range(1, len(gru_history["val_f1"]) + 1), gru_history["val_f1"], marker="o", label="GRU")
ax.set_title("LSTM vs GRU — Validation Macro-F1 per Epoch", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Macro-F1")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "09_lstm_vs_gru_val_f1.png")
plt.show()


### 5.5 Section 5 Summary

- Implemented a GloVe-initialized **GRU classifier**, architecturally
  matched to the Section 4 LSTM (same embedding, hidden size, dropout,
  training budget) for a fair cell-type comparison.
- Reused the identical weighted-loss `run_training` pipeline — same
  optimizer, scheduler, gradient clipping, early stopping, and checkpointing
  behavior as the LSTM.
- Saved the best checkpoint to `models/gru_best.pt` and history to
  `reports/gru_history.json`.
- Plotted GRU learning curves (`figures/08_gru_learning_curves.png`) and an
  early LSTM-vs-GRU validation-F1 comparison
  (`figures/09_lstm_vs_gru_val_f1.png`) — full test-set metrics follow in
  Section 8.

**Next: Section 6 — BiLSTM + Attention Model** (custom additive attention
layer, bidirectional context, attention-weight extraction for later
visualization). Waiting for confirmation to proceed.


## Section 6 — BiLSTM + Attention Model

Our strongest classical architecture: a **Bidirectional LSTM** with a
**custom additive (Bahdanau-style) attention layer** built from scratch in
PyTorch, replacing the "last hidden state" pooling used in Sections 4-5
with a learned, per-token weighted sum over the *entire* BiLSTM output
sequence.

This lets the model:
- Attend to **emotion-bearing words anywhere in the sentence**, not just the
  end
- Produce an **interpretable attention distribution** per prediction, which
  we visualize directly in Section 9


In [ ]:
# 6.1 — Reload artifacts (safe standalone execution)
with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
VOCAB_SIZE = len(itos)
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX = stoi[PAD_TOKEN]

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")
EMBEDDING_DIM = embedding_matrix.shape[1]

npz = np.load(DATA_DIR / "sequence_encoded.npz")
X_train_seq, y_train = npz["X_train"], npz["y_train"]
X_val_seq, y_val = npz["X_val"], npz["y_val"]
X_test_seq, y_test = npz["X_test"], npz["y_test"]

with open(REPORTS_DIR / "class_weights.json") as f:
    class_weight_report = json.load(f)
CLASS_WEIGHTS = torch.tensor(
    [class_weight_report[ID2LABEL[i]] for i in range(NUM_CLASSES)], dtype=torch.float32
).to(DEVICE)

from torch.utils.data import Dataset, DataLoader


class EmotionSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64
train_loader = DataLoader(EmotionSequenceDataset(X_train_seq, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionSequenceDataset(X_val_seq, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(EmotionSequenceDataset(X_test_seq, y_test), batch_size=BATCH_SIZE, shuffle=False)
print(f"Vocab: {VOCAB_SIZE:,} | Classes: {NUM_CLASSES} | Train batches: {len(train_loader)}")


### 6.2 Custom Additive Attention Layer

Given BiLSTM outputs $H \in \mathbb{R}^{T \times 2h}$ (one vector per
token), the attention layer computes:

$$u_t = \tanh(W H_t + b), \qquad
\alpha_t = \frac{\exp(u_t^\top v)}{\sum_{t'=1}^{T} \exp(u_{t'}^\top v)}, \qquad
c = \sum_{t=1}^{T} \alpha_t H_t$$

where $W$, $b$, and the context vector $v$ are **learned parameters**, $\alpha_t$
is the attention weight for token $t$ (summing to 1 across the sequence),
and $c$ is the resulting context vector fed to the classifier. Padding
positions are masked to $-\infty$ before the softmax so they receive
exactly zero weight.


In [ ]:
# 6.2.1 — Custom attention layer, implemented from scratch
class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention over a sequence of hidden states."""

    def __init__(self, hidden_dim: int, attention_dim: int = 128):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor, mask: torch.Tensor):
        """
        hidden_states: (batch, seq_len, hidden_dim)
        mask: (batch, seq_len) — 1 for real tokens, 0 for padding
        Returns:
            context: (batch, hidden_dim)
            attn_weights: (batch, seq_len)
        """
        u = torch.tanh(self.W(hidden_states))              # (batch, seq_len, attn_dim)
        scores = self.v(u).squeeze(-1)                       # (batch, seq_len)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn_weights = torch.softmax(scores, dim=1)           # (batch, seq_len)
        context = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)  # (batch, hidden_dim)
        return context, attn_weights


In [ ]:
# 6.2.2 — BiLSTM + Attention classifier
class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, attention_dim=128,
                 num_classes=6, pad_idx=0, dropout=0.3, freeze_embeddings=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze_embeddings,
            padding_idx=pad_idx,
        )
        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.attention = AdditiveAttention(hidden_dim=hidden_dim * 2, attention_dim=attention_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, return_attention: bool = False):
        mask = (x != self.pad_idx).long()                    # (batch, seq_len)
        embedded = self.embedding(x)                          # (batch, seq_len, embed_dim)
        lstm_out, _ = self.bilstm(embedded)                    # (batch, seq_len, 2*hidden_dim)
        context, attn_weights = self.attention(lstm_out, mask)
        out = self.dropout(context)
        logits = self.fc(out)
        if return_attention:
            return logits, attn_weights
        return logits


bilstm_attn_model = BiLSTMAttentionClassifier(
    embedding_matrix, hidden_dim=128, attention_dim=128, num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX, dropout=0.3, freeze_embeddings=False,
).to(DEVICE)

n_params = sum(p.numel() for p in bilstm_attn_model.parameters() if p.requires_grad)
print(bilstm_attn_model)
print(f"\nTrainable parameters: {n_params:,}")


In [ ]:
# 6.2.3 — Shape / sanity check on a single batch, including attention weights
sample_xb, sample_yb = next(iter(train_loader))
sample_xb = sample_xb.to(DEVICE)
with torch.no_grad():
    logits, attn = bilstm_attn_model(sample_xb, return_attention=True)

print(f"logits shape: {logits.shape}")           # (batch, num_classes)
print(f"attention weights shape: {attn.shape}")   # (batch, seq_len)
print(f"attention weights sum to 1 per example: {torch.allclose(attn.sum(dim=1), torch.ones(attn.size(0), device=DEVICE), atol=1e-4)}")


### 6.3 Training

The training loop must be adapted slightly: the model's `forward()` now
optionally returns attention weights, so we call it in standard (logits-only)
mode during training/evaluation, and only request attention weights
explicitly in Section 9.


In [ ]:
# 6.3.1 — Reusable training utility (identical to Sections 4.4 / 5.3;
# works unchanged since forward() defaults to return_attention=False)
import time
from sklearn.metrics import f1_score, accuracy_score


def run_training(model, train_loader, val_loader, model_name: str,
                  num_epochs=15, lr=1e-3, weight_decay=1e-5,
                  class_weights=None, patience=3, grad_clip=5.0):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

    history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    best_val_f1 = -1.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"{model_name}_best.pt"

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        model.train()
        train_losses, train_preds, train_targets = [], [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.extend(logits.argmax(dim=1).cpu().numpy())
            train_targets.extend(yb.cpu().numpy())

        train_loss = np.mean(train_losses)
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        model.eval()
        val_losses, val_preds, val_targets = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(yb.cpu().numpy())

        val_loss = np.mean(val_losses)
        val_f1 = f1_score(val_targets, val_preds, average="macro")
        val_acc = accuracy_score(val_targets, val_preds)

        scheduler.step(val_f1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        elapsed = time.time() - start_time
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
              f"train_f1={train_f1:.4f} val_f1={val_f1:.4f} val_acc={val_acc:.4f} | "
              f"{elapsed:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch} (no improvement for {patience} epochs).")
                break

    model.load_state_dict(torch.load(best_path))
    print(f"\nBest val_f1={best_val_f1:.4f} — restored best checkpoint from {best_path}")
    return model, history


In [ ]:
# 6.3.2 — Train the BiLSTM + Attention model
bilstm_attn_model, bilstm_attn_history = run_training(
    bilstm_attn_model, train_loader, val_loader, model_name="bilstm_attention",
    num_epochs=15, lr=1e-3, class_weights=CLASS_WEIGHTS, patience=3,
)

with open(REPORTS_DIR / "bilstm_attention_history.json", "w") as f:
    json.dump(bilstm_attn_history, f, indent=2)


### 6.4 Learning Curves & Three-Way Comparison Preview


In [ ]:
# 6.4.1 — Plot loss / F1 curves for BiLSTM + Attention
def plot_learning_curves(history: dict, model_name: str, save_name: str):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    epochs_range = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
    axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(epochs_range, history["train_f1"], marker="o", label="Train Macro-F1")
    axes[1].plot(epochs_range, history["val_f1"], marker="o", label="Val Macro-F1")
    axes[1].set_title(f"{model_name} — Macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].legend()

    plt.suptitle(f"{model_name} Learning Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name)
    plt.show()


plot_learning_curves(bilstm_attn_history, "BiLSTM + Attention", "10_bilstm_attention_learning_curves.png")


In [ ]:
# 6.4.2 — Three-way validation Macro-F1 comparison (preview; full test-set comparison in Section 8)
with open(REPORTS_DIR / "lstm_history.json") as f:
    lstm_history = json.load(f)
with open(REPORTS_DIR / "gru_history.json") as f:
    gru_history = json.load(f)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(lstm_history["val_f1"]) + 1), lstm_history["val_f1"], marker="o", label="LSTM")
ax.plot(range(1, len(gru_history["val_f1"]) + 1), gru_history["val_f1"], marker="o", label="GRU")
ax.plot(range(1, len(bilstm_attn_history["val_f1"]) + 1), bilstm_attn_history["val_f1"], marker="o", label="BiLSTM + Attention")
ax.set_title("Validation Macro-F1 per Epoch — All Recurrent Models", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Macro-F1")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "11_all_recurrent_val_f1.png")
plt.show()


### 6.5 Section 6 Summary

- Implemented a **custom additive (Bahdanau) attention layer from scratch**
  (`AdditiveAttention`) — no library shortcut — with explicit padding
  masking so `<PAD>` tokens receive exactly zero attention weight.
- Built the **BiLSTMAttentionClassifier**: `Embedding (GloVe) → BiLSTM →
  Attention → Dropout → Linear`, with an optional `return_attention=True`
  forward path used later for interpretability.
- Verified attention weights are a valid probability distribution
  (`sum(attn) == 1` per example).
- Trained with the same reusable pipeline and class-weighted loss as
  Sections 4-5; saved the best checkpoint to
  `models/bilstm_attention_best.pt`.
- Plotted learning curves and a three-way (LSTM / GRU / BiLSTM+Attention)
  validation-F1 comparison — saved to `figures/10` and `figures/11`.

**Next: Section 7 — DistilBERT Fine-Tuning** (Hugging Face `Trainer`,
`AutoModelForSequenceClassification`, using the pre-tokenized data cached in
Section 2.8). Waiting for confirmation to proceed.


## Section 7 — DistilBERT Fine-Tuning

Our strongest model: fine-tuning **`distilbert-base-uncased`** (pretrained
Transformer, distilled from BERT-base) end-to-end on the 6-class emotion
task, using the Hugging Face `Trainer` API. We reuse the tokenized tensors
cached in Section 2.8 (`data/bert_encoded.pt`).


In [ ]:
# 7.1 — Reload artifacts (safe standalone execution)
with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

with open(REPORTS_DIR / "class_weights.json") as f:
    class_weight_report = json.load(f)
CLASS_WEIGHTS = torch.tensor(
    [class_weight_report[ID2LABEL[i]] for i in range(NUM_CLASSES)], dtype=torch.float32
)

bert_data = torch.load(DATA_DIR / "bert_encoded.pt")
print("Cached BERT-encoded splits:", list(bert_data.keys()))
print("Train tensor keys:", list(bert_data["train"].keys()))
print(f"Train size: {bert_data['train']['labels'].shape[0]:,}")
print(f"Val size:   {bert_data['val']['labels'].shape[0]:,}")
print(f"Test size:  {bert_data['test']['labels'].shape[0]:,}")


### 7.2 Dataset Wrapper for the Hugging Face `Trainer`


In [ ]:
# 7.2.1 — Torch Dataset wrapping the cached tokenized tensors
from torch.utils.data import Dataset


class BertEmotionDataset(Dataset):
    def __init__(self, encodings: dict):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels = encodings["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


bert_train_ds = BertEmotionDataset(bert_data["train"])
bert_val_ds = BertEmotionDataset(bert_data["val"])
bert_test_ds = BertEmotionDataset(bert_data["test"])
print(f"Train: {len(bert_train_ds):,} | Val: {len(bert_val_ds):,} | Test: {len(bert_test_ds):,}")


### 7.3 Model Initialization

`AutoModelForSequenceClassification` attaches a fresh linear classification
head (6 outputs) on top of the pretrained DistilBERT encoder. We pass
`id2label` / `label2id` so the saved model is self-describing.


In [ ]:
# 7.3.1 — Load pretrained DistilBERT with a 6-way classification head
from transformers import AutoModelForSequenceClassification, AutoTokenizer

BERT_MODEL_NAME = "distilbert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(DEVICE)

n_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


### 7.4 Custom Weighted-Loss Trainer

We subclass `Trainer` to apply the same **class-weighted cross-entropy
loss** used by the recurrent models, keeping the loss function consistent
across every architecture in this project.


In [ ]:
# 7.4.1 — Weighted-loss Trainer subclass
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device) if class_weights is not None else None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, NUM_CLASSES), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "f1_macro": f1_macro, "precision_macro": precision, "recall_macro": recall}


### 7.5 Training Configuration & Fine-Tuning


In [ ]:
# 7.5.1 — TrainingArguments
BERT_OUTPUT_DIR = str(MODELS_DIR / "distilbert_checkpoints")

training_args = TrainingArguments(
    output_dir=BERT_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir=str(MODELS_DIR / "distilbert_logs"),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = WeightedLossTrainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_ds,
    eval_dataset=bert_val_ds,
    compute_metrics=compute_metrics,
    class_weights=CLASS_WEIGHTS,
)


In [ ]:
# 7.5.2 — Fine-tune DistilBERT
train_result = trainer.train()
print(train_result)


In [ ]:
# 7.5.3 — Persist the fine-tuned model, tokenizer, and training log
FINAL_BERT_DIR = MODELS_DIR / "distilbert_emotion_final"
trainer.save_model(str(FINAL_BERT_DIR))
bert_tokenizer.save_pretrained(str(FINAL_BERT_DIR))

log_history = trainer.state.log_history
with open(REPORTS_DIR / "distilbert_training_log.json", "w") as f:
    json.dump(log_history, f, indent=2)

print(f"Saved fine-tuned DistilBERT to: {FINAL_BERT_DIR}")


### 7.6 Training Curves


In [ ]:
# 7.6.1 — Extract per-epoch train/eval loss and eval F1 from the Trainer log
train_logs = [e for e in log_history if "loss" in e and "eval_loss" not in e]
eval_logs = [e for e in log_history if "eval_loss" in e]

train_epochs = [e["epoch"] for e in train_logs]
train_losses = [e["loss"] for e in train_logs]
eval_epochs = [e["epoch"] for e in eval_logs]
eval_losses = [e["eval_loss"] for e in eval_logs]
eval_f1s = [e["eval_f1_macro"] for e in eval_logs]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(train_epochs, train_losses, label="Train Loss (per logging step)", alpha=0.6)
axes[0].plot(eval_epochs, eval_losses, marker="o", label="Val Loss (per epoch)")
axes[0].set_title("DistilBERT — Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(eval_epochs, eval_f1s, marker="o", color="darkorange", label="Val Macro-F1")
axes[1].set_title("DistilBERT — Validation Macro F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()

plt.suptitle("DistilBERT Fine-Tuning Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "12_distilbert_learning_curves.png")
plt.show()


### 7.7 Section 7 Summary

- Fine-tuned **`distilbert-base-uncased`** end-to-end on the 6-class task
  using the Hugging Face `Trainer` API, reusing the tokenized tensors cached
  in Section 2.8 — no re-tokenization needed.
- Implemented a **`WeightedLossTrainer`** subclass so DistilBERT uses the
  exact same class-weighted cross-entropy loss as every recurrent model in
  this project, keeping the comparison in Section 8 fair.
- Trained with a standard fine-tuning recipe: `lr=2e-5`, linear warmup,
  weight decay, mixed precision (`fp16`) on GPU, best-model selection by
  validation macro-F1.
- Saved the final fine-tuned model + tokenizer to
  `models/distilbert_emotion_final/` (loadable directly via
  `AutoModelForSequenceClassification.from_pretrained(...)`), and the full
  training log to `reports/distilbert_training_log.json`.
- Plotted fine-tuning curves to `figures/12_distilbert_learning_curves.png`.

**Next: Section 8 — Evaluation & Model Comparison** (unified test-set
confusion matrices, classification reports, and a final comparison table
across all four models). Waiting for confirmation to proceed.


## Section 8 — Evaluation & Model Comparison

Now that all four models are trained, we evaluate every one of them on the
**same held-out test set** with the **same metrics**, producing:

1. Per-model confusion matrices
2. Per-model classification reports (precision / recall / F1 per class)
3. A unified model comparison table
4. A grouped bar chart comparing all models across key metrics


In [ ]:
# 8.1 — Reload every artifact needed for evaluation (safe standalone execution)
with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
VOCAB_SIZE = len(itos)
PAD_IDX = stoi["<PAD>"]

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")

npz = np.load(DATA_DIR / "sequence_encoded.npz")
X_test_seq, y_test = npz["X_test"], npz["y_test"]

bert_data = torch.load(DATA_DIR / "bert_encoded.pt")

from torch.utils.data import Dataset, DataLoader


class EmotionSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


test_loader = DataLoader(EmotionSequenceDataset(X_test_seq, y_test), batch_size=128, shuffle=False)
print(f"Test set size: {len(y_test):,}")


### 8.2 Reload Model Architectures & Best Checkpoints


In [ ]:
# 8.2.1 — Re-declare architectures (identical to Sections 4-6) and load best weights
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(self.dropout(hidden[-1]))


class GRUClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.gru = nn.GRU(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        return self.fc(self.dropout(hidden[-1]))


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=128):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, hidden_states, mask):
        u = torch.tanh(self.W(hidden_states))
        scores = self.v(u).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)
        return context, attn_weights


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, attention_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.bilstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True, bidirectional=True)
        self.attention = AdditiveAttention(hidden_dim * 2, attention_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, return_attention=False):
        mask = (x != self.pad_idx).long()
        embedded = self.embedding(x)
        lstm_out, _ = self.bilstm(embedded)
        context, attn_weights = self.attention(lstm_out, mask)
        logits = self.fc(self.dropout(context))
        if return_attention:
            return logits, attn_weights
        return logits


lstm_model = LSTMClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
lstm_model.load_state_dict(torch.load(MODELS_DIR / "lstm_best.pt", map_location=DEVICE))

gru_model = GRUClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
gru_model.load_state_dict(torch.load(MODELS_DIR / "gru_best.pt", map_location=DEVICE))

bilstm_attn_model = BiLSTMAttentionClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
bilstm_attn_model.load_state_dict(torch.load(MODELS_DIR / "bilstm_attention_best.pt", map_location=DEVICE))

print("Loaded LSTM, GRU, and BiLSTM+Attention best checkpoints.")


In [ ]:
# 8.2.2 — Load fine-tuned DistilBERT
from transformers import AutoModelForSequenceClassification, AutoTokenizer

FINAL_BERT_DIR = MODELS_DIR / "distilbert_emotion_final"
bert_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_BERT_DIR))
bert_model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_BERT_DIR)).to(DEVICE)
bert_model.eval()
print("Loaded fine-tuned DistilBERT.")


### 8.3 Unified Prediction & Evaluation Utilities


In [ ]:
# 8.3.1 — Get predictions for a recurrent (sequence) model
@torch.no_grad()
def predict_sequence_model(model, data_loader):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    for xb, yb in data_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().numpy())
        all_targets.extend(yb.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_targets), np.array(all_preds), np.array(all_probs)


# 8.3.2 — Get predictions for DistilBERT
@torch.no_grad()
def predict_bert_model(model, encodings, batch_size=128):
    model.eval()
    input_ids, attention_mask, labels = encodings["input_ids"], encodings["attention_mask"], encodings["labels"]
    all_preds, all_probs = [], []
    for i in range(0, len(labels), batch_size):
        batch_ids = input_ids[i:i + batch_size].to(DEVICE)
        batch_mask = attention_mask[i:i + batch_size].to(DEVICE)
        logits = model(input_ids=batch_ids, attention_mask=batch_mask).logits
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return labels.numpy(), np.array(all_preds), np.array(all_probs)


In [ ]:
# 8.3.3 — Run inference for all four models on the test set
y_true_lstm, y_pred_lstm, y_prob_lstm = predict_sequence_model(lstm_model, test_loader)
y_true_gru, y_pred_gru, y_prob_gru = predict_sequence_model(gru_model, test_loader)
y_true_bilstm, y_pred_bilstm, y_prob_bilstm = predict_sequence_model(bilstm_attn_model, test_loader)
y_true_bert, y_pred_bert, y_prob_bert = predict_bert_model(bert_model, bert_data["test"])

model_predictions = {
    "LSTM": (y_true_lstm, y_pred_lstm),
    "GRU": (y_true_gru, y_pred_gru),
    "BiLSTM + Attention": (y_true_bilstm, y_pred_bilstm),
    "DistilBERT": (y_true_bert, y_pred_bert),
}
print("Inference complete for all 4 models.")


### 8.4 Confusion Matrices


In [ ]:
# 8.4.1 — Confusion matrix grid (all 4 models)
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, (model_name, (y_true, y_pred)) in zip(axes, model_predictions.items()):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=TARGET_EMOTIONS, yticklabels=TARGET_EMOTIONS,
                ax=ax, cbar=False, vmin=0, vmax=1)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Normalized Confusion Matrices — Test Set", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "13_confusion_matrices_all_models.png")
plt.show()


### 8.5 Classification Reports


In [ ]:
# 8.5.1 — Per-model classification reports
from sklearn.metrics import classification_report

classification_reports = {}
for model_name, (y_true, y_pred) in model_predictions.items():
    report_str = classification_report(y_true, y_pred, target_names=TARGET_EMOTIONS, digits=3, zero_division=0)
    report_dict = classification_report(y_true, y_pred, target_names=TARGET_EMOTIONS, digits=3,
                                         zero_division=0, output_dict=True)
    classification_reports[model_name] = report_dict
    print(f"{'=' * 70}\n{model_name}\n{'=' * 70}")
    print(report_str)

with open(REPORTS_DIR / "classification_reports.json", "w") as f:
    json.dump(classification_reports, f, indent=2)
print("\nSaved all classification reports to reports/classification_reports.json")


### 8.6 Model Comparison Table


In [ ]:
# 8.6.1 — Build the unified comparison table
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

comparison_rows = []
for model_name, (y_true, y_pred) in model_predictions.items():
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted"),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Model").round(4)
comparison_df = comparison_df.sort_values("Macro F1", ascending=False)
comparison_df.to_csv(REPORTS_DIR / "model_comparison_table.csv")
comparison_df


In [ ]:
# 8.6.2 — Visualize the comparison table as a grouped bar chart
fig, ax = plt.subplots(figsize=(11, 6))
comparison_df[["Accuracy", "Macro F1", "Weighted F1"]].plot(
    kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e", "#2ca02c"], edgecolor="black", linewidth=0.6,
)
ax.set_title("Model Comparison — Test Set Metrics", fontsize=14, fontweight="bold")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "14_model_comparison_bar_chart.png")
plt.show()


In [ ]:
# 8.6.3 — Per-class Macro-F1 heatmap across models (which emotions are hardest, per model)
per_class_f1 = pd.DataFrame({
    model_name: [classification_reports[model_name][emotion]["f1-score"] for emotion in TARGET_EMOTIONS]
    for model_name in model_predictions
}, index=TARGET_EMOTIONS)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(per_class_f1, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "F1-score"})
ax.set_title("Per-Class F1-Score by Model", fontsize=14, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("Emotion")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "15_per_class_f1_heatmap.png")
plt.show()


### 8.7 Section 8 Summary

- Ran unified test-set inference for **all four models** (LSTM, GRU,
  BiLSTM+Attention, DistilBERT) through shared prediction utilities.
- Generated **normalized confusion matrices** for every model in a single
  comparison grid (`figures/13_confusion_matrices_all_models.png`).
- Printed and saved full **classification reports** (precision/recall/F1 per
  emotion) to `reports/classification_reports.json`.
- Built the final **model comparison table**
  (`reports/model_comparison_table.csv`) ranked by macro-F1, and visualized
  it as a grouped bar chart.
- Produced a **per-class F1 heatmap** across models, showing which emotions
  are hardest for which architecture (typically `surprise` and `fear`
  are hardest due to lower support / semantic overlap with `joy`/`disgust`
  respectively — confirm against your own run's numbers).

**Next: Section 9 — Attention Visualization** (extracting and plotting
per-token attention weights from the BiLSTM+Attention model on real test
examples). Waiting for confirmation to proceed.


## Section 9 — Attention Visualization

The whole point of the custom `AdditiveAttention` layer built in Section 6
is **interpretability**: we can extract, per test example, exactly how much
weight the BiLSTM+Attention model placed on each token when making its
prediction. This section visualizes that directly.


In [ ]:
# 9.1 — Reload artifacts (safe standalone execution)
with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
PAD_IDX = stoi["<PAD>"]
UNK_TOKEN = "<UNK>"

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Re-declare architecture and load best weights (identical to Section 6/8)
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=128):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, hidden_states, mask):
        u = torch.tanh(self.W(hidden_states))
        scores = self.v(u).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)
        return context, attn_weights


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, attention_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.bilstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True, bidirectional=True)
        self.attention = AdditiveAttention(hidden_dim * 2, attention_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, return_attention=False):
        mask = (x != self.pad_idx).long()
        embedded = self.embedding(x)
        lstm_out, _ = self.bilstm(embedded)
        context, attn_weights = self.attention(lstm_out, mask)
        logits = self.fc(self.dropout(context))
        if return_attention:
            return logits, attn_weights
        return logits


bilstm_attn_model = BiLSTMAttentionClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
bilstm_attn_model.load_state_dict(torch.load(MODELS_DIR / "bilstm_attention_best.pt", map_location=DEVICE))
bilstm_attn_model.eval()
print("Loaded BiLSTM+Attention model for interpretability analysis.")


### 9.2 Attention Extraction Utility

Given a raw sentence, this function tokenizes and pads it exactly like
Section 2's pipeline, runs it through the model with
`return_attention=True`, and returns the predicted emotion alongside
per-token attention weights (trimmed back to the real, non-padded tokens).


In [ ]:
# 9.2.1 — Predict + extract attention weights for a single raw sentence
@torch.no_grad()
def predict_with_attention(text: str, model, stoi: dict, max_len: int):
    tokens = text.lower().split()[:max_len]
    n_real_tokens = len(tokens)
    ids = [stoi.get(tok, stoi[UNK_TOKEN]) for tok in tokens]
    ids = ids + [stoi["<PAD>"]] * (max_len - len(ids))

    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    logits, attn_weights = model(x, return_attention=True)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(probs.argmax())

    attn = attn_weights.cpu().numpy()[0][:n_real_tokens]
    attn = attn / attn.sum()  # renormalize over real tokens only, for display

    return {
        "tokens": tokens,
        "attention": attn,
        "predicted_emotion": ID2LABEL[pred_id],
        "probs": {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)},
    }


# Quick test
example_result = predict_with_attention("I am so incredibly happy and excited today!", bilstm_attn_model, stoi, MAX_SEQ_LEN)
print(f"Predicted: {example_result['predicted_emotion']}")
for tok, w in zip(example_result["tokens"], example_result["attention"]):
    print(f"  {tok:15s} {w:.3f}")


### 9.3 Attention Heatmap Visualization

A reusable plotting function that renders per-token attention as a color-
coded horizontal strip, making it immediately visible which words drove the
model's prediction.


In [ ]:
# 9.3.1 — Attention heatmap plot
def plot_attention_heatmap(result: dict, true_label: str = None, ax=None, save_path=None):
    tokens, attn = result["tokens"], result["attention"]
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.6), 2.2))

    attn_2d = attn.reshape(1, -1)
    sns.heatmap(attn_2d, annot=False, cmap="OrRd", cbar=standalone, ax=ax,
                xticklabels=tokens, yticklabels=False, vmin=0, vmax=attn.max())
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=10)

    title = f"Predicted: {result['predicted_emotion'].capitalize()}"
    if true_label is not None:
        title += f"  |  Actual: {true_label.capitalize()}"
    ax.set_title(title, fontsize=11, fontweight="bold")

    if standalone:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path)
        plt.show()


plot_attention_heatmap(example_result, save_path=FIGURES_DIR / "16_attention_example_single.png")


### 9.4 Attention on Real Test Examples (One Per Emotion)

We sample one correctly-classified test example per target emotion and
visualize attention for all six side-by-side — a strong, presentation-ready
qualitative demonstration of what the model has learned to focus on.


In [ ]:
# 9.4.1 — Sample one correctly-predicted example per emotion and plot attention grid
random.seed(SEED)
selected_examples = []

for emotion in TARGET_EMOTIONS:
    candidates = test_df[test_df["emotion"] == emotion]["clean_text"].tolist()
    random.shuffle(candidates)
    for text in candidates:
        result = predict_with_attention(text, bilstm_attn_model, stoi, MAX_SEQ_LEN)
        if result["predicted_emotion"] == emotion and 3 <= len(result["tokens"]) <= 12:
            selected_examples.append((emotion, text, result))
            break

fig, axes = plt.subplots(len(selected_examples), 1, figsize=(10, 2.3 * len(selected_examples)))
for ax, (emotion, text, result) in zip(axes, selected_examples):
    plot_attention_heatmap(result, true_label=emotion, ax=ax)

plt.suptitle("Attention Visualization — One Correctly-Classified Example per Emotion",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "17_attention_grid_all_emotions.png", bbox_inches="tight")
plt.show()


### 9.5 Top Attended Words per Emotion (Aggregate Analysis)

Beyond single examples, we aggregate attention weights across the **entire
test set** to find which words the model attends to most, on average, when
predicting each emotion — a global, corpus-level interpretability view.


In [ ]:
# 9.5.1 — Aggregate top attended tokens per predicted emotion across the test set
from collections import defaultdict

token_attention_sum = defaultdict(lambda: defaultdict(float))
token_attention_count = defaultdict(lambda: defaultdict(int))

sample_size = min(2000, len(test_df))  # cap for reasonable runtime
test_sample = test_df.sample(n=sample_size, random_state=SEED)

for text in test_sample["clean_text"]:
    result = predict_with_attention(text, bilstm_attn_model, stoi, MAX_SEQ_LEN)
    pred = result["predicted_emotion"]
    for tok, w in zip(result["tokens"], result["attention"]):
        token_attention_sum[pred][tok] += w
        token_attention_count[pred][tok] += 1

top_attended = {}
for emotion in TARGET_EMOTIONS:
    avg_scores = {
        tok: token_attention_sum[emotion][tok] / token_attention_count[emotion][tok]
        for tok in token_attention_sum[emotion]
        if token_attention_count[emotion][tok] >= 3  # ignore rare one-off tokens
    }
    top_attended[emotion] = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)[:10]

for emotion, words in top_attended.items():
    print(f"\n{emotion.upper()}:")
    for tok, score in words:
        print(f"  {tok:15s} avg_attention={score:.3f}")


In [ ]:
# 9.5.2 — Visualize top attended words per emotion
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

for ax, emotion in zip(axes, TARGET_EMOTIONS):
    words, scores = zip(*top_attended[emotion])
    sns.barplot(x=list(scores), y=list(words), palette="rocket", ax=ax)
    ax.set_title(emotion.capitalize(), fontsize=12, fontweight="bold")
    ax.set_xlabel("Avg. Attention Weight")

plt.suptitle("Top Attended Words per Emotion (Aggregated over Test Set)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "18_top_attended_words_per_emotion.png")
plt.show()


### 9.6 Section 9 Summary

- Built a reusable `predict_with_attention()` utility that runs the
  BiLSTM+Attention model on raw text and returns per-token attention
  weights alongside the prediction.
- Visualized attention as color-coded heatmaps for individual examples, and
  produced a **one-example-per-emotion grid** showing exactly which words
  drove each correct classification (`figures/17`).
- Performed a **corpus-level aggregation**: average attention weight per
  token, grouped by predicted emotion, revealing the words the model relies
  on most globally (e.g., strong exclamations for `anger`/`surprise`,
  positive adjectives for `joy`) — saved to `figures/18`.
- This directly validates the motivation for using attention over plain
  LSTM/GRU pooling: the model's decisions are now **inspectable**, not a
  black box.

**Next: Section 10 — Gradio Deployment** (interactive demo app serving all
four trained models, with attention visualization built into the UI).
Waiting for confirmation to proceed.


## Section 10 — Gradio Deployment

We package all four trained models into a single interactive **Gradio**
web app: the user types a sentence, selects a model, and gets back the
predicted emotion, a full probability distribution over all six classes,
and — for the BiLSTM+Attention model — a live attention heatmap.


In [ ]:
# 10.1 — Reload every model + artifact needed for the demo (safe standalone execution)
import gradio as gr

with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
PAD_IDX = stoi["<PAD>"]
UNK_TOKEN = "<UNK>"

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")

EMOTION_COLORS = {
    "joy": "#FFD700", "sadness": "#4682B4", "anger": "#DC143C",
    "fear": "#8A2BE2", "surprise": "#FF8C00", "disgust": "#2E8B57",
}
print("Artifacts loaded for deployment.")


In [ ]:
# 10.2 — Re-declare all four architectures and load best checkpoints
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(self.dropout(hidden[-1]))


class GRUClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.gru = nn.GRU(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        return self.fc(self.dropout(hidden[-1]))


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=128):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, hidden_states, mask):
        u = torch.tanh(self.W(hidden_states))
        scores = self.v(u).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)
        return context, attn_weights


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, attention_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.bilstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True, bidirectional=True)
        self.attention = AdditiveAttention(hidden_dim * 2, attention_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, return_attention=False):
        mask = (x != self.pad_idx).long()
        embedded = self.embedding(x)
        lstm_out, _ = self.bilstm(embedded)
        context, attn_weights = self.attention(lstm_out, mask)
        logits = self.fc(self.dropout(context))
        if return_attention:
            return logits, attn_weights
        return logits


from transformers import AutoModelForSequenceClassification, AutoTokenizer

lstm_model = LSTMClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
lstm_model.load_state_dict(torch.load(MODELS_DIR / "lstm_best.pt", map_location=DEVICE))
lstm_model.eval()

gru_model = GRUClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
gru_model.load_state_dict(torch.load(MODELS_DIR / "gru_best.pt", map_location=DEVICE))
gru_model.eval()

bilstm_attn_model = BiLSTMAttentionClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
bilstm_attn_model.load_state_dict(torch.load(MODELS_DIR / "bilstm_attention_best.pt", map_location=DEVICE))
bilstm_attn_model.eval()

FINAL_BERT_DIR = MODELS_DIR / "distilbert_emotion_final"
bert_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_BERT_DIR))
bert_model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_BERT_DIR)).to(DEVICE)
bert_model.eval()

print("All 4 models loaded and ready for deployment.")


### 10.3 Unified Inference Function

A single function routes the input text to whichever model the user
selects, applies the correct preprocessing/tokenization for that model
family, and returns both a probability dictionary (for Gradio's `Label`
component) and an attention HTML string (populated only for
BiLSTM+Attention).


In [ ]:
# 10.3.1 — Shared text cleaning (identical to Section 2.2, self-contained here)
URL_RE = re.compile(r"http\S+|www\.\S+")
BRACKET_TOKEN_RE = re.compile(r"\[(NAME|RELIGION)\]")
HTML_ENTITY_RE = re.compile(r"&amp;|&lt;|&gt;|&#x200B;")
MULTI_SPACE_RE = re.compile(r"\s+")
REPEATED_CHAR_RE = re.compile(r"(.)\1{2,}")
ALLOWED_CHARS_RE = re.compile(r"[^a-zA-Z0-9\s.,!?'\-]")


def clean_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = BRACKET_TOKEN_RE.sub(" ", text)
    text = HTML_ENTITY_RE.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = ALLOWED_CHARS_RE.sub(" ", text)
    text = REPEATED_CHAR_RE.sub(r"\1\1", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text


def attention_to_html(tokens: list, weights: np.ndarray) -> str:
    """Render tokens as background-shaded spans proportional to attention weight."""
    if len(weights) == 0:
        return "<i>No tokens to display.</i>"
    max_w = weights.max() if weights.max() > 0 else 1.0
    spans = []
    for tok, w in zip(tokens, weights):
        intensity = w / max_w
        alpha = 0.15 + 0.75 * intensity
        spans.append(
            f'<span style="background-color: rgba(255, 99, 71, {alpha:.2f}); '
            f'padding: 3px 5px; margin: 2px; border-radius: 4px; display: inline-block;">{tok}</span>'
        )
    return '<div style="line-height: 2.4; font-size: 16px;">' + " ".join(spans) + "</div>"


In [ ]:
# 10.3.2 — Unified prediction router
@torch.no_grad()
def predict_emotion(text: str, model_choice: str):
    if not text or not text.strip():
        return {}, "<i>Enter some text above.</i>"

    cleaned = clean_text(text)

    if model_choice == "DistilBERT (best)":
        enc = bert_tokenizer(cleaned, padding="max_length", truncation=True,
                              max_length=64, return_tensors="pt").to(DEVICE)
        logits = bert_model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        prob_dict = {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)}
        return prob_dict, "<i>Attention visualization is only available for the BiLSTM + Attention model.</i>"

    tokens = cleaned.lower().split()[:MAX_SEQ_LEN]
    n_real = len(tokens)
    ids = [stoi.get(t, stoi[UNK_TOKEN]) for t in tokens]
    ids = ids + [stoi["<PAD>"]] * (MAX_SEQ_LEN - len(ids))
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    if model_choice == "LSTM":
        logits = lstm_model(x)
        attn_html = "<i>Attention visualization is only available for the BiLSTM + Attention model.</i>"
    elif model_choice == "GRU":
        logits = gru_model(x)
        attn_html = "<i>Attention visualization is only available for the BiLSTM + Attention model.</i>"
    else:  # "BiLSTM + Attention"
        logits, attn_weights = bilstm_attn_model(x, return_attention=True)
        attn = attn_weights.cpu().numpy()[0][:n_real]
        attn = attn / attn.sum() if attn.sum() > 0 else attn
        attn_html = attention_to_html(tokens, attn)

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    prob_dict = {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    return prob_dict, attn_html


# Quick smoke test before launching the UI
test_probs, test_html = predict_emotion("This is the best day of my life, I can't stop smiling!", "BiLSTM + Attention")
print(test_probs)


### 10.4 Gradio Interface


In [ ]:
# 10.4.1 — Build and launch the Gradio Blocks app
EXAMPLE_INPUTS = [
    ["I am so incredibly happy and excited about this news!", "BiLSTM + Attention"],
    ["I feel so lonely and heartbroken since she left.", "DistilBERT (best)"],
    ["This is absolutely disgusting, I can't believe they served this.", "BiLSTM + Attention"],
    ["I can't believe you actually did that, are you serious right now?!", "GRU"],
    ["I'm terrified of what might happen next.", "LSTM"],
    ["Wow, I did not see that coming at all!", "BiLSTM + Attention"],
]

with gr.Blocks(title="Emotion Classification with Attention") as demo:
    gr.Markdown(
        """
        # 🎭 Emotion Classification with Attention
        Classify short text into **Joy, Sadness, Anger, Fear, Surprise, or Disgust**
        using one of four trained models. Select **BiLSTM + Attention** to see
        which words the model focused on to make its decision.
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Enter text",
                placeholder="Type a sentence describing how someone feels...",
                lines=3,
            )
            model_dropdown = gr.Dropdown(
                choices=["LSTM", "GRU", "BiLSTM + Attention", "DistilBERT (best)"],
                value="BiLSTM + Attention",
                label="Select Model",
            )
            predict_btn = gr.Button("Predict Emotion", variant="primary")
            gr.Examples(examples=EXAMPLE_INPUTS, inputs=[text_input, model_dropdown])

        with gr.Column(scale=2):
            label_output = gr.Label(label="Predicted Emotion Probabilities", num_top_classes=6)
            gr.Markdown("### Attention Visualization")
            attention_output = gr.HTML(label="Attention Weights")

    predict_btn.click(fn=predict_emotion, inputs=[text_input, model_dropdown], outputs=[label_output, attention_output])
    text_input.submit(fn=predict_emotion, inputs=[text_input, model_dropdown], outputs=[label_output, attention_output])

demo.launch(share=True, debug=False)


> **Note:** `share=True` generates a temporary public URL valid for 72
> hours, useful for demoing the app from a Kaggle notebook (which has no
> persistent public hosting). For a permanent deployment, export this cell's
> logic into a standalone `app.py` and deploy to **Hugging Face Spaces**
> (see the README for a ready-to-use `app.py` template).

### 10.5 Section 10 Summary

- Built a **unified inference router** that dispatches to the correct model
  family (recurrent vs. Transformer) with the correct preprocessing for
  each.
- Implemented an **HTML attention renderer** that shades each token's
  background proportional to its attention weight, for a self-contained,
  dependency-free visualization inside the Gradio UI.
- Assembled a full **Gradio Blocks app**: text input, model selector,
  probability `Label` output, live attention HTML, and a curated set of
  example inputs spanning all six emotions.
- Launched with `share=True` for an instant public demo link directly from
  Kaggle.

**Next: Section 11 — Conclusion** (final write-up, model comparison
discussion, limitations, future work) **and the GitHub-ready README**.
Waiting for confirmation to proceed.


## Section 10 — Gradio Deployment

An interactive **Gradio** app that lets a user type any sentence and get:

1. A **model selector** (LSTM / GRU / BiLSTM+Attention / DistilBERT)
2. The **predicted emotion** with a full probability bar chart
3. An **attention heatmap** (shown automatically when BiLSTM+Attention is
   selected)

This ties together every artifact produced so far into a single, runnable
demo — the kind of thing you'd link from a GitHub README or a portfolio
page.


In [ ]:
# 10.1 — Reload every artifact needed for the demo (safe standalone execution)
import gradio as gr
import matplotlib
matplotlib.use("Agg")

with open(DATA_DIR / "vocab.json") as f:
    vocab_data = json.load(f)
stoi, itos = vocab_data["stoi"], vocab_data["itos"]
MAX_SEQ_LEN = vocab_data["max_seq_len"]
PAD_IDX = stoi["<PAD>"]
UNK_TOKEN = "<UNK>"

with open(REPORTS_DIR / "label_mapping.json") as f:
    label_map = json.load(f)
LABEL2ID = label_map["LABEL2ID"]
ID2LABEL = {int(k): v for k, v in label_map["ID2LABEL"].items()}
TARGET_EMOTIONS = list(LABEL2ID.keys())
NUM_CLASSES = len(TARGET_EMOTIONS)

embedding_matrix = np.load(DATA_DIR / "glove_embedding_matrix.npy")
print("Artifacts reloaded for deployment.")


In [ ]:
# 10.2 — Re-declare all four architectures and load their best weights
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(self.dropout(hidden[-1]))


class GRUClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.gru = nn.GRU(embedding_matrix.shape[1], hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        return self.fc(self.dropout(hidden[-1]))


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=128):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, hidden_states, mask):
        u = torch.tanh(self.W(hidden_states))
        scores = self.v(u).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)
        return context, attn_weights


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, attention_dim=128, num_classes=6, pad_idx=0, dropout=0.3):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), padding_idx=pad_idx)
        self.bilstm = nn.LSTM(embedding_matrix.shape[1], hidden_dim, batch_first=True, bidirectional=True)
        self.attention = AdditiveAttention(hidden_dim * 2, attention_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, return_attention=False):
        mask = (x != self.pad_idx).long()
        embedded = self.embedding(x)
        lstm_out, _ = self.bilstm(embedded)
        context, attn_weights = self.attention(lstm_out, mask)
        logits = self.fc(self.dropout(context))
        if return_attention:
            return logits, attn_weights
        return logits


from transformers import AutoModelForSequenceClassification, AutoTokenizer

lstm_model = LSTMClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
lstm_model.load_state_dict(torch.load(MODELS_DIR / "lstm_best.pt", map_location=DEVICE))
lstm_model.eval()

gru_model = GRUClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
gru_model.load_state_dict(torch.load(MODELS_DIR / "gru_best.pt", map_location=DEVICE))
gru_model.eval()

bilstm_attn_model = BiLSTMAttentionClassifier(embedding_matrix, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(DEVICE)
bilstm_attn_model.load_state_dict(torch.load(MODELS_DIR / "bilstm_attention_best.pt", map_location=DEVICE))
bilstm_attn_model.eval()

FINAL_BERT_DIR = MODELS_DIR / "distilbert_emotion_final"
bert_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_BERT_DIR))
bert_model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_BERT_DIR)).to(DEVICE)
bert_model.eval()

print("All four models loaded and ready for inference.")


### 10.3 Unified Inference Function

A single `predict_emotion()` function that dispatches to the correct model
based on the dropdown selection, returning a probability dict (rendered by
Gradio's `Label` component as a bar chart) plus an attention plot (only
populated for BiLSTM+Attention).


In [ ]:
# 10.3.1 — Preprocessing (mirrors Section 2.2 cleaning + Section 2.5 encoding)
def clean_text_for_inference(text: str) -> str:
    text = URL_RE.sub(" ", text) if 'URL_RE' in globals() else text
    return text.strip()


def encode_for_recurrent_models(text: str):
    tokens = text.lower().split()[:MAX_SEQ_LEN]
    ids = [stoi.get(tok, stoi[UNK_TOKEN]) for tok in tokens]
    ids = ids + [stoi["<PAD>"]] * (MAX_SEQ_LEN - len(ids))
    return torch.tensor([ids], dtype=torch.long).to(DEVICE), tokens


@torch.no_grad()
def predict_emotion(text: str, model_choice: str):
    if not text or not text.strip():
        return {}, None

    if model_choice == "DistilBERT":
        enc = bert_tokenizer(text, return_tensors="pt", truncation=True,
                              padding="max_length", max_length=64).to(DEVICE)
        logits = bert_model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        prob_dict = {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)}
        return prob_dict, None

    x, tokens = encode_for_recurrent_models(text)

    if model_choice == "LSTM":
        logits = lstm_model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        return {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)}, None

    if model_choice == "GRU":
        logits = gru_model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        return {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)}, None

    if model_choice == "BiLSTM + Attention":
        logits, attn_weights = bilstm_attn_model(x, return_attention=True)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        prob_dict = {ID2LABEL[i]: float(probs[i]) for i in range(NUM_CLASSES)}

        n_real = len(tokens)
        attn = attn_weights.cpu().numpy()[0][:n_real]
        attn = attn / attn.sum()

        fig, ax = plt.subplots(figsize=(max(6, n_real * 0.6), 2.2))
        attn_2d = attn.reshape(1, -1)
        sns.heatmap(attn_2d, cmap="OrRd", cbar=False, ax=ax,
                    xticklabels=tokens, yticklabels=False, vmin=0, vmax=attn.max())
        ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=10)
        ax.set_title("Attention Weights", fontsize=11, fontweight="bold")
        plt.tight_layout()
        return prob_dict, fig

    raise ValueError(f"Unknown model choice: {model_choice}")


# Quick sanity check before launching the UI
probs, _ = predict_emotion("I can't believe this happened, I'm so scared!", "BiLSTM + Attention")
print(probs)


### 10.4 Gradio Interface


In [ ]:
# 10.4.1 — Build and launch the Gradio app
EXAMPLE_INPUTS = [
    ["I am so incredibly happy right now, this is the best day ever!", "BiLSTM + Attention"],
    ["I can't stop crying, I miss him so much.", "DistilBERT"],
    ["This is absolutely disgusting, I can't believe they served this.", "LSTM"],
    ["I was not expecting that twist at all, what a shock!", "GRU"],
    ["Get out of my way right now, I am furious with you!", "BiLSTM + Attention"],
]

with gr.Blocks(title="Emotion Classification with Attention") as demo:
    gr.Markdown(
        """
        # Emotion Classification with Attention
        Type a sentence, pick a model, and see the predicted emotion.
        Select **BiLSTM + Attention** to also see which words the model
        focused on.
        """
    )
    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Enter text",
                placeholder="e.g. I can't believe I won the lottery!",
                lines=3,
            )
            model_dropdown = gr.Dropdown(
                choices=["LSTM", "GRU", "BiLSTM + Attention", "DistilBERT"],
                value="BiLSTM + Attention",
                label="Model",
            )
            predict_btn = gr.Button("Predict Emotion", variant="primary")
        with gr.Column(scale=2):
            label_output = gr.Label(label="Predicted Emotion Probabilities", num_top_classes=6)
            attention_plot = gr.Plot(label="Attention Weights (BiLSTM + Attention only)")

    predict_btn.click(fn=predict_emotion, inputs=[text_input, model_dropdown],
                       outputs=[label_output, attention_plot])
    text_input.submit(fn=predict_emotion, inputs=[text_input, model_dropdown],
                       outputs=[label_output, attention_plot])

    gr.Examples(examples=EXAMPLE_INPUTS, inputs=[text_input, model_dropdown])

demo.launch(share=True, debug=False)


### 10.5 Section 10 Summary

- Reloaded all four trained models from their saved checkpoints into a
  single inference session.
- Built a unified `predict_emotion()` dispatcher that routes to the correct
  model and returns both a probability distribution and (for
  BiLSTM+Attention) a live attention heatmap.
- Assembled an interactive **Gradio Blocks** UI with a model selector, text
  input, probability bar chart, attention plot, and preset examples.
- Launched with `share=True`, producing a public shareable link directly
  from the Kaggle notebook — ready to drop into a README or portfolio page.

**Next: Section 11 — Professional Conclusion + GitHub-Ready README** (final
write-up summarizing findings, limitations, and future work, plus a
complete `README.md` for the GitHub repo). Waiting for confirmation to
proceed.


## Section 11 — Conclusion

### 11.1 Project Recap

This project built a complete, end-to-end pipeline for **6-class emotion
classification** (`joy, sadness, anger, fear, surprise, disgust`) on the
GoEmotions dataset, spanning:

- Data acquisition, cleaning, and a leakage-free stratified split
- A from-scratch vocabulary + GloVe embedding matrix for classical recurrent
  models
- Four model architectures of increasing sophistication: **LSTM → GRU →
  BiLSTM + custom Attention → fine-tuned DistilBERT**
- Rigorous, identical evaluation of all four on a held-out test set
- Model-level and corpus-level **attention interpretability**
- A deployable, interactive **Gradio** demo

### 11.2 Key Findings

*(Fill in with your own run's numbers from `reports/model_comparison_table.csv`
after executing the notebook — this section is intentionally written so you
insert real results here rather than fabricated ones.)*

- Which model achieved the highest macro-F1 on the test set, and by how much?
- Did attention meaningfully outperform plain last-hidden-state pooling
  (BiLSTM+Attention vs. LSTM/GRU)? Refer to `figures/14_model_comparison_bar_chart.png`.
- Which emotions were hardest across all models (see
  `figures/15_per_class_f1_heatmap.png`) — typically classes with fewer
  training examples (check `figures/01_class_distribution.png`) or semantic
  overlap (e.g., `fear` vs. `sadness`) underperform.
- Did the attention visualizations in Section 9 line up with human
  intuition about which words carry emotional signal?

### 11.3 Limitations

- **Single-label assumption**: real text is often multi-emotional; this
  project intentionally simplified to single-label classification by
  filtering GoEmotions, which discards a large fraction of the original
  corpus (see the retention rate in `reports/01_eda_report.json`) and may
  not reflect real-world multi-label distributions.
- **Domain specificity**: GoEmotions is Reddit-comment text; performance may
  not transfer directly to other domains (e.g., customer support tickets,
  clinical notes) without re-training or domain adaptation.
- **Class imbalance**: despite class-weighted loss, rarer emotions
  (typically `fear`, `surprise`, `disgust`) likely still underperform more
  frequent ones (`joy`, `sadness`) — see the class distribution from
  Section 1.
- **Attention ≠ full explanation**: attention weights indicate *where the
  model looked*, not a complete causal account of *why* it predicted a
  given label; they are a useful but partial interpretability tool.

### 11.4 Future Work

- Extend to the **full multi-label** GoEmotions setting (27 emotions +
  neutral) using sigmoid outputs + per-class thresholds instead of softmax.
- Try **self-attention / Transformer encoders trained from scratch** (not
  just fine-tuned DistilBERT) to isolate the effect of pretraining vs.
  architecture.
- Apply **data augmentation** (back-translation, synonym replacement) to
  boost minority-class performance without discarding data.
- **Quantize / distill** the fine-tuned DistilBERT model further for
  lower-latency production deployment.
- Add **calibration analysis** (e.g., reliability diagrams) to check whether
  predicted probabilities are trustworthy, not just accurate at the top-1
  level.

### 11.5 Reproducibility

Every intermediate artifact needed to reproduce or extend this project is
saved under `/kaggle/working/`:

| Directory | Contents |
|---|---|
| `data/` | Cleaned/split CSVs, vocabulary, GloVe matrix, cached tokenized tensors |
| `models/` | Best checkpoints for all four models |
| `figures/` | All 18 saved plots (EDA, learning curves, confusion matrices, attention) |
| `reports/` | JSON/CSV reports: EDA stats, GloVe coverage, class weights, classification reports, comparison table |

All random seeds are fixed (`SEED = 42`) for reproducibility, though exact
numeric results may vary slightly across hardware/library versions.
